# Phát Hiện Bất Thường Theo Luật Nghiệp Vụ dựa trên Session người dùng trong web thương mại điện tử

## RetailRocket E-Commerce Dataset

Dataset: https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset

Đơn vị phân tích chính là `session_id`, được tạo từ `visitorid` và khoảng nghỉ 30 phút giữa hai sự kiện liên tiếp. Dataset không có nhãn bất thường thật, vì vậy nhãn `is_anomaly_rule` và `anomaly_types` là pseudo-label do nhóm thiết kế bằng luật nghiệp vụ.

## Hệ thống business rules và anomaly types

| Mã | Anomaly type | Mô tả nghiệp vụ | Dấu hiệu chính trong notebook |
| --- | --- | --- | --- |
| BR01 | Bot scraper | Session có hành vi duyệt tự động hoặc quét dữ liệu nhanh. | Nhiều event trong session hoặc tốc độ event/phút cao. |
| BR02 | Ghost buyer | Có giao dịch nhưng không có bước thêm giỏ trong cùng session. | `transaction > 0` và `addtocart = 0`. |
| BR03 | Click fraud | Xem nhiều sản phẩm nhưng không có ý định mua hoặc thêm giỏ. | Nhiều `view`, không có `addtocart`, không có `transaction`. |
| BR04 | Rapid-fire | Các thao tác diễn ra quá sát nhau, giống bot/script. | Có event liên tiếp dưới 1 giây hoặc tỉ lệ rapid-fire cao. |
| BR05 | Night crawler | Session hoạt động chủ yếu vào khung giờ đêm. | Tỉ lệ event 0h-5h cao và session đủ số event tối thiểu. |
| BR06 | Item hoarding | Thêm cùng một sản phẩm vào giỏ nhiều lần trong một session. | `max_same_item_atc` vượt ngưỡng. |
| BR07 | Session bomb | Session xem/quét quá nhiều item khác nhau. | `unique_items` cao. |
| BR08 | Sequence violation | Thứ tự hành vi bất thường: mua trước khi xem/thêm giỏ cùng item. | `transaction` xuất hiện trước `view/addtocart` của cùng item trong session. |
| BR09 | Transaction burst | Session có số lượng giao dịch cao bất thường. | `n_transaction` vượt ngưỡng. |
| BR10 | Cart abandonment | Thêm nhiều sản phẩm vào giỏ nhưng không giao dịch. | `n_addtocart` cao và `n_transaction = 0`. |
| BR11 | Repeated view spam | Xem lặp lại cùng một item quá nhiều lần. | `max_same_item_view` vượt ngưỡng. |
| BR12 | Category scanning | Quét nhiều nhóm danh mục sản phẩm trong một session. | `unique_categories` cao nếu dữ liệu category khả dụng. |

Các rule này dùng để gắn nhãn multi-label cho session. Một session có thể vừa là `Bot scraper`, vừa là `Rapid-fire`, vừa là `Session bomb`. Các thuật toán ML dùng pseudo-label này để so sánh/mô phỏng, không được diễn giải như nhãn gian lận thật.

Phần mô hình được chia thành `train`, `validation`, `test`. Validation dùng để chọn threshold dự đoán; test dùng để báo cáo kết quả cuối.


## Phân công thuật toán

| Thành viên | Thuật toán | Loại | Vai trò trong đồ án |
| --- | --- | --- | --- |
| Đình Tuấn | XGBoost | Phân loại có giám sát | Mô hình boosting mạnh, xử lý dữ liệu phi tuyến và mất cân bằng lớp |
| Lê Văn Anh | Decision Tree | Phân loại có giám sát | Mô hình dễ giải thích, minh họa được luật quyết định |
| Tuấn Anh | Random Forest | Phân loại có giám sát | Ensemble nhiều cây, ổn định hơn cây đơn và có feature importance |
| Thủy | LightGBM | Phân loại có giám sát | Gradient boosting leaf-wise, nhanh và hiệu quả trên dữ liệu dạng bảng; so sánh thêm với XGBoost |
| Đức Anh | Isolation Forest | Phát hiện bất thường không giám sát | Thuật toán chuyên biệt cho anomaly detection |

## 1. Import và cấu hình

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    hamming_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

try:
    from xgboost import XGBClassifier
except Exception as import_error:
    raise ImportError(
        "Chua nap duoc XGBoost. Cai package bang `.venv/bin/python -m pip install xgboost`; "
        "tren macOS can them OpenMP runtime bang `brew install libomp`."
    ) from import_error

try:
    from lightgbm import LGBMClassifier
except Exception as import_error:
    raise ImportError(
        "Chua nap duoc LightGBM. Cai package bang `.venv/bin/python -m pip install lightgbm`; "
        "tren macOS neu gap loi OpenMP thi cai `brew install libomp`."
    ) from import_error

warnings.filterwarnings('ignore')

RANDOM_STATE = 10
OUTPUT_DIR = Path('.')
DATA_PATH = Path('./data/events.csv')
SESSION_GAP_SEC = 30 * 60

MAX_SUPERVISED_ROWS = 3_000_000
IFOREST_TRAIN_ROWS = 250_000
PLOT_SAMPLE_ROWS = 30_000

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
COLORS = {
    'normal': '#2E7D32',
    'anomaly': '#C62828',
    'warning': '#EF6C00',
    'primary': '#1565C0',
    'secondary': '#6A1B9A',
    'dark': '#263238',
    'muted': '#607D8B',
}

print('Thu vien va cau hinh da san sang')


## 2. Tải dữ liệu, tiền xử lý và tạo session

Session được định nghĩa bằng khoảng nghỉ 30 phút giữa hai event liên tiếp của cùng `visitorid`.

In [ ]:
print('Đang tải dữ liệu events.csv...')
events = pd.read_csv(DATA_PATH)

required_columns = {'timestamp', 'visitorid', 'event', 'itemid', 'transactionid'}
missing_columns = required_columns.difference(events.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

rows_before_dedup = len(events)
events = events.drop_duplicates().reset_index(drop=True)
duplicate_count = rows_before_dedup - len(events)

events = events[events['event'].isin(['view', 'addtocart', 'transaction'])].copy()

events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['datetime'].dt.date
events['hour'] = events['datetime'].dt.hour
events['dayofweek'] = events['datetime'].dt.dayofweek

events = events.sort_values(['visitorid', 'timestamp', 'event', 'itemid']).reset_index(drop=True)
events['prev_timestamp'] = events.groupby('visitorid')['timestamp'].shift(1)
events['time_diff_sec'] = (events['timestamp'] - events['prev_timestamp']) / 1000

is_new_session = events['time_diff_sec'].isna() | (events['time_diff_sec'] > SESSION_GAP_SEC)
events['session_number'] = is_new_session.groupby(events['visitorid']).cumsum().astype('int32')
events['session_id'] = events['visitorid'].astype(str) + '_S' + events['session_number'].astype(str)

events['prev_session_timestamp'] = events.groupby('session_id')['timestamp'].shift(1)
events['time_diff_session_sec'] = (events['timestamp'] - events['prev_session_timestamp']) / 1000

session_count = events['session_id'].nunique()
print(f'Số dòng trùng lặp đã xóa: {duplicate_count:,}')
print(f'Tổng số sự kiện sau tiền xử lý: {len(events):,}')
print(f'Số visitor duy nhất: {events["visitorid"].nunique():,}')
print(f'Số session tạo ra: {session_count:,}')
print(f'Trung bình sự kiện/session: {len(events) / session_count:.2f}')
print(f'Khoảng thời gian: {events["datetime"].min()} -> {events["datetime"].max()}')
print('\nPhân phối loại sự kiện:')
display(events['event'].value_counts().rename_axis('event').reset_index(name='count'))

In [ ]:
session_sizes = events.groupby('session_id').size()
session_duration = (events.groupby('session_id')['timestamp'].max() - events.groupby('session_id')['timestamp'].min()) / 1000

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

order = events['event'].value_counts().index
sns.countplot(data=events, x='event', order=order, ax=axes[0], palette='Set2')
axes[0].set_title('Phân phối loại sự kiện')
axes[0].set_xlabel('Loại sự kiện')
axes[0].set_ylabel('Số lượng')
axes[0].bar_label(axes[0].containers[0], fmt='{:,.0f}')

sns.histplot(session_sizes.clip(upper=20), bins=20, color=COLORS['primary'], ax=axes[1])
axes[1].set_title('Số sự kiện mỗi phiên (giới hạn ≤ 20)')
axes[1].set_xlabel('Số sự kiện/phiên')
axes[1].set_ylabel('Số phiên')

sns.histplot((session_duration / 60).clip(upper=30), bins=30, color=COLORS['secondary'], ax=axes[2])
axes[2].set_title('Thời lượng mỗi phiên (phút, giới hạn ≤ 30)')
axes[2].set_xlabel('Phút')
axes[2].set_ylabel('Số phiên')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_session_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Xây dựng session behavior profile

Mỗi dòng của bảng profile tương ứng với một `session_id`. `visitorid` chỉ còn là metadata để truy vết người dùng.

In [ ]:
print('Đang xây dựng session behavior profile...')
session_events = events.copy()
session_groups = session_events.groupby('session_id', sort=False)

session_profile = pd.DataFrame(index=session_groups.size().index)
session_profile.index.name = 'session_id'
session_profile['visitorid'] = session_groups['visitorid'].first().astype('int64')
session_profile['session_number'] = session_groups['session_number'].first().astype('int32')
session_profile['session_start'] = session_groups['datetime'].min()
session_profile['session_end'] = session_groups['datetime'].max()
session_profile['session_start_hour'] = session_profile['session_start'].dt.hour
session_profile['session_dayofweek'] = session_profile['session_start'].dt.dayofweek
session_profile['total_events'] = session_groups.size().astype('int32')
session_profile['unique_items'] = session_groups['itemid'].nunique().astype('int32')
session_profile['active_hours'] = session_groups['hour'].nunique().astype('int16')
session_profile['session_duration_sec'] = ((session_groups['timestamp'].max() - session_groups['timestamp'].min()) / 1000).astype('float32')

event_counts = session_events.groupby(['session_id', 'event']).size().unstack(fill_value=0)
event_counts.columns = [f'n_{column}' for column in event_counts.columns]
for column in ['n_view', 'n_addtocart', 'n_transaction']:
    if column not in event_counts.columns:
        event_counts[column] = 0
session_profile = session_profile.join(event_counts[['n_view', 'n_addtocart', 'n_transaction']].astype('int32'))
session_profile['unique_event_types'] = (session_profile[['n_view', 'n_addtocart', 'n_transaction']] > 0).sum(axis=1).astype('int8')

event_probabilities = event_counts[['n_view', 'n_addtocart', 'n_transaction']].div(session_profile['total_events'].clip(lower=1), axis=0)
session_profile['event_type_entropy'] = (-(event_probabilities.replace(0, np.nan) * np.log2(event_probabilities.replace(0, np.nan))).sum(axis=1)).fillna(0).astype('float32')

hour_counts = session_events.groupby(['session_id', 'hour']).size().unstack(fill_value=0)
hour_probabilities = hour_counts.div(hour_counts.sum(axis=1).clip(lower=1), axis=0)
session_profile['hour_entropy'] = (-(hour_probabilities.replace(0, np.nan) * np.log2(hour_probabilities.replace(0, np.nan))).sum(axis=1)).fillna(0).astype('float32')

events_with_interval = session_events.dropna(subset=['time_diff_session_sec'])
interval_stats = events_with_interval.groupby('session_id')['time_diff_session_sec'].agg(
    min_interval_sec='min',
    mean_interval_sec='mean',
    median_interval_sec='median',
    std_interval_sec='std',
    max_interval_sec='max',
)
session_profile = session_profile.join(interval_stats).fillna({
    'min_interval_sec': 0,
    'mean_interval_sec': 0,
    'median_interval_sec': 0,
    'std_interval_sec': 0,
    'max_interval_sec': 0,
})
interval_columns = ['min_interval_sec', 'mean_interval_sec', 'median_interval_sec', 'std_interval_sec', 'max_interval_sec']
session_profile[interval_columns] = session_profile[interval_columns].astype('float32')

session_profile['rapid_fire_count'] = events_with_interval[events_with_interval['time_diff_session_sec'] < 1].groupby('session_id').size().reindex(session_profile.index, fill_value=0).astype('int16')
session_profile['night_events'] = session_events[session_events['hour'].between(0, 5)].groupby('session_id').size().reindex(session_profile.index, fill_value=0).astype('int16')
session_profile['peak_events'] = session_events[session_events['hour'].between(9, 21)].groupby('session_id').size().reindex(session_profile.index, fill_value=0).astype('int16')
session_profile['weekend_events'] = session_events[session_events['dayofweek'].isin([5, 6])].groupby('session_id').size().reindex(session_profile.index, fill_value=0).astype('int16')

view_events = session_events[session_events['event'] == 'view']
if len(view_events) > 0:
    max_same_item_view = view_events.groupby(['session_id', 'itemid']).size().groupby('session_id').max()
    session_profile['max_same_item_view'] = max_same_item_view.reindex(session_profile.index, fill_value=0).astype('int16')
else:
    session_profile['max_same_item_view'] = 0

addtocart_events = session_events[session_events['event'] == 'addtocart']
if len(addtocart_events) > 0:
    max_same_item_addtocart = addtocart_events.groupby(['session_id', 'itemid']).size().groupby('session_id').max()
    session_profile['max_same_item_atc'] = max_same_item_addtocart.reindex(session_profile.index, fill_value=0).astype('int16')
else:
    session_profile['max_same_item_atc'] = 0

session_profile['duration_min'] = (session_profile['session_duration_sec'] / 60).clip(lower=1).astype('float32')
session_profile['events_per_minute'] = (session_profile['total_events'] / session_profile['duration_min']).astype('float32')
session_profile['view_rate'] = (session_profile['n_view'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['atc_rate'] = (session_profile['n_addtocart'] / session_profile['n_view'].clip(lower=1)).astype('float32')
session_profile['buy_rate'] = (session_profile['n_transaction'] / session_profile['n_view'].clip(lower=1)).astype('float32')
session_profile['night_ratio'] = (session_profile['night_events'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['peak_ratio'] = (session_profile['peak_events'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['weekend_ratio'] = (session_profile['weekend_events'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['rapid_ratio'] = (session_profile['rapid_fire_count'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['items_per_event'] = (session_profile['unique_items'] / session_profile['total_events'].clip(lower=1)).astype('float32')
session_profile['view_to_cart_ratio'] = (session_profile['n_view'] / session_profile['n_addtocart'].clip(lower=1)).astype('float32')
session_profile['cart_to_transaction_ratio'] = (session_profile['n_addtocart'] / session_profile['n_transaction'].clip(lower=1)).astype('float32')

# Dac trung dem so su kien vi pham trinh tu (transaction truoc view/addtocart cung item) - input cua BR08.
# Co so: bot/script co the bo qua buoc xem/them gio binh thuong (Tan & Kumar, 2002, ve hanh vi dieu huong bat thuong).
_interaction_event = session_events['event'].isin(['view', 'addtocart']).astype('int8')
_prior_item_interactions = _interaction_event.groupby([session_events['session_id'], session_events['itemid']]).cumsum()
_seq_violation = session_events.loc[
    (session_events['event'] == 'transaction') & (_prior_item_interactions == 0)
].groupby('session_id').size()
session_profile['n_sequence_violation_events'] = _seq_violation.reindex(session_profile.index, fill_value=0).astype('int16')

session_profile['unique_categories'] = 0
session_profile['unique_parent_categories'] = 0
try:
    # Doc theo chunk va loc property=='categoryid' ngay tren tung chunk de an toan bo nho
    # (hai file item_properties rat lon ~880MB). Chi giu lai cac dong categoryid.
    def _load_category_ids(path):
        frames = []
        for chunk in pd.read_csv(path, usecols=['itemid', 'property', 'value'],
                                 dtype={'property': 'string'}, chunksize=2_000_000):
            part = chunk.loc[chunk['property'] == 'categoryid', ['itemid', 'value']]
            if len(part):
                frames.append(part)
        if frames:
            return pd.concat(frames, ignore_index=True)
        return pd.DataFrame(columns=['itemid', 'value'])

    item_categories = pd.concat(
        [_load_category_ids('./data/item_properties_part1.csv'),
         _load_category_ids('./data/item_properties_part2.csv')],
        ignore_index=True,
    )
    item_categories['categoryid'] = pd.to_numeric(item_categories['value'], errors='coerce')
    item_categories = item_categories.dropna(subset=['categoryid']).drop_duplicates('itemid')
    item_categories['categoryid'] = item_categories['categoryid'].astype('int64')

    category_tree = pd.read_csv('./data/category_tree.csv')
    category_parent = category_tree[['categoryid', 'parentid']].drop_duplicates('categoryid')
    item_categories = item_categories.merge(category_parent, on='categoryid', how='left')

    session_items = session_events[['session_id', 'itemid']].drop_duplicates()
    session_items = session_items.merge(item_categories[['itemid', 'categoryid', 'parentid']], on='itemid', how='left')
    session_profile['unique_categories'] = session_items.groupby('session_id')['categoryid'].nunique().reindex(session_profile.index, fill_value=0).astype('int16')
    session_profile['unique_parent_categories'] = session_items.groupby('session_id')['parentid'].nunique().reindex(session_profile.index, fill_value=0).astype('int16')
    print('Đã bổ sung đặc trưng danh mục từ item_properties và category_tree')
except Exception as category_error:
    print(f'Bỏ qua đặc trưng danh mục vì không nạp được item properties: {category_error}')

print(f'Kích thước session profile: {session_profile.shape}')
print(f'session_id là duy nhất: {session_profile.index.is_unique}')
display(session_profile.head())

In [ ]:
summary_cols = [
    'total_events', 'unique_items', 'unique_categories', 'unique_parent_categories',
    'session_duration_sec', 'events_per_minute', 'n_view', 'n_addtocart', 'n_transaction',
    'rapid_fire_count', 'rapid_ratio', 'night_ratio', 'max_same_item_view', 'max_same_item_atc',
    'event_type_entropy', 'hour_entropy',
]
profile_summary = session_profile[summary_cols].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T
print('Thống kê các đặc trưng session chính:')
display(profile_summary)

## 4. Business rules và multi-label anomaly tagging theo session

Các luật gắn nhãn hành vi đáng nghi ở cấp session. Một session có thể mang nhiều anomaly type cùng lúc.

### Cơ sở của các ngưỡng (có trích dẫn)

Các ngưỡng không chọn tùy tiện mà neo vào tài liệu về web usage mining và bot/fraud detection:

| Tham số | Giá trị | Cơ sở & nguồn |
| --- | --- | --- |
| Ngưỡng tách session | 30 phút không hoạt động | Chuẩn de-facto của web usage mining, bắt nguồn từ Catledge & Pitkow (1995) (trung bình 9.3 phút giữa các event + 1.5 SD ≈ 25.5 phút, làm tròn 30); chuẩn hóa bởi Cooley, Mobasher & Srivastava (1999) và Google Analytics. |
| Bot scraper: số event / phiên cao (≥12) hoặc tốc độ ≥8 event/phút | bot quét nhanh, nhiều request | Tan & Kumar (2002): session dài và tốc độ request cao là dấu hiệu robot; bot "cố thu thập nhiều thông tin nhanh nhất có thể". |
| Session bomb: ≥12 item khác nhau | quét nhiều sản phẩm | Tan & Kumar (2002): robot có xu hướng phủ phần lớn site trong một phiên (nhiều resource khác nhau). |
| Rapid-fire: khoảng cách event < 1 giây | thao tác phi nhân lực | Khoảng cách giữa hai thao tác của người ~0.2–0.4s, của bot gần 0–0.05s; tốc độ dưới ngưỡng vật lý của con người ⇒ script (bot-detection literature). |
| Cart abandonment: ≥10 addtocart, 0 transaction | bỏ giỏ | Baymard Institute (2025): tỉ lệ bỏ giỏ trung bình ~70% (meta-analysis 50 nghiên cứu) ⇒ bỏ giỏ là phổ biến, ngưỡng đặt cao để chỉ bắt trường hợp cực đoan. |
| Night crawler: ≥75% event lúc 0–5h | hoạt động ban đêm bất thường | Phân tích tương phản thời gian hoạt động robot vs người (Doran & Gokhale, 2011). |
| Sequence violation | transaction trước view/addtocart cùng item | Vi phạm thứ tự hành vi tự nhiên của người mua; dựa trên phân tích navigational pattern (Tan & Kumar, 2002). |

Đầy đủ trích dẫn ở mục **Tài liệu tham khảo** cuối notebook.


In [ ]:
THRESHOLDS = {
    'BR01_min_events_bot': 12,
    'BR01_min_events_per_minute': 8,
    'BR01_min_events_for_speed': 5,
    'BR03_min_views_for_click_fraud': 6,
    'BR04_rapid_fire_count': 1,
    'BR04_rapid_fire_ratio': 0.20,
    'BR04_min_events_for_rapid': 3,
    'BR05_night_ratio': 0.75,
    'BR05_min_events_for_night': 4,
    'BR06_max_same_item_atc': 1,
    'BR07_min_unique_items_session_bomb': 12,
    'BR09_min_transactions_burst': 3,
    'BR10_min_addtocart_abandonment': 10,
    'BR11_max_same_item_view': 20,
    'BR12_min_unique_categories': 20,
}

sessions = session_profile.copy()

sessions['flag_BR01_bot_scraper'] = (
    (sessions['total_events'] >= THRESHOLDS['BR01_min_events_bot'])
    | (
        (sessions['events_per_minute'] >= THRESHOLDS['BR01_min_events_per_minute'])
        & (sessions['total_events'] >= THRESHOLDS['BR01_min_events_for_speed'])
    )
).astype(int)

sessions['flag_BR02_ghost_buyer'] = (
    (sessions['n_transaction'] > 0) & (sessions['n_addtocart'] == 0)
).astype(int)

sessions['flag_BR03_click_fraud'] = (
    (sessions['n_view'] >= THRESHOLDS['BR03_min_views_for_click_fraud'])
    & (sessions['n_addtocart'] == 0)
    & (sessions['n_transaction'] == 0)
).astype(int)

sessions['flag_BR04_rapid_fire'] = (
    (
        (sessions['rapid_fire_count'] >= THRESHOLDS['BR04_rapid_fire_count'])
        | (sessions['rapid_ratio'] >= THRESHOLDS['BR04_rapid_fire_ratio'])
    )
    & (sessions['total_events'] >= THRESHOLDS['BR04_min_events_for_rapid'])
).astype(int)

sessions['flag_BR05_night_crawler'] = (
    (sessions['night_ratio'] >= THRESHOLDS['BR05_night_ratio'])
    & (sessions['total_events'] >= THRESHOLDS['BR05_min_events_for_night'])
).astype(int)

sessions['flag_BR06_item_hoarding'] = (
    sessions['max_same_item_atc'] > THRESHOLDS['BR06_max_same_item_atc']
).astype(int)

sessions['flag_BR07_session_bomb'] = (
    sessions['unique_items'] >= THRESHOLDS['BR07_min_unique_items_session_bomb']
).astype(int)

interaction_event = session_events['event'].isin(['view', 'addtocart']).astype('int8')
prior_item_interactions = interaction_event.groupby([session_events['session_id'], session_events['itemid']]).cumsum()
sequence_violation_sessions = session_events.loc[
    (session_events['event'] == 'transaction') & (prior_item_interactions == 0),
    'session_id',
].unique()
sessions['flag_BR08_sequence_violation'] = sessions.index.isin(sequence_violation_sessions).astype(int)

sessions['flag_BR09_transaction_burst'] = (
    sessions['n_transaction'] >= THRESHOLDS['BR09_min_transactions_burst']
).astype(int)

sessions['flag_BR10_cart_abandonment'] = (
    (sessions['n_addtocart'] >= THRESHOLDS['BR10_min_addtocart_abandonment'])
    & (sessions['n_transaction'] == 0)
).astype(int)

sessions['flag_BR11_repeated_view_spam'] = (
    sessions['max_same_item_view'] >= THRESHOLDS['BR11_max_same_item_view']
).astype(int)

sessions['flag_BR12_category_scanning'] = (
    sessions['unique_categories'] >= THRESHOLDS['BR12_min_unique_categories']
).astype(int)

anomaly_type_map = {
    'flag_BR01_bot_scraper': 'Bot scraper',
    'flag_BR02_ghost_buyer': 'Ghost buyer',
    'flag_BR03_click_fraud': 'Click fraud',
    'flag_BR04_rapid_fire': 'Rapid-fire',
    'flag_BR05_night_crawler': 'Night crawler',
    'flag_BR06_item_hoarding': 'Item hoarding',
    'flag_BR07_session_bomb': 'Session bomb',
    'flag_BR08_sequence_violation': 'Sequence violation',
    'flag_BR09_transaction_burst': 'Transaction burst',
    'flag_BR10_cart_abandonment': 'Cart abandonment',
    'flag_BR11_repeated_view_spam': 'Repeated view spam',
    'flag_BR12_category_scanning': 'Category scanning',
}

flag_cols = list(anomaly_type_map.keys())
sessions['total_flags'] = sessions[flag_cols].sum(axis=1)
sessions['is_anomaly_rule'] = (sessions['total_flags'] > 0).astype(int)
sessions['is_anomaly'] = sessions['is_anomaly_rule']

flag_matrix = sessions[flag_cols].to_numpy(dtype=bool)
anomaly_type_labels = np.array(list(anomaly_type_map.values()), dtype=object)
sessions['anomaly_type_count'] = flag_matrix.sum(axis=1).astype(int)
sessions['anomaly_types'] = [
    ', '.join(anomaly_type_labels[row_flags].tolist()) if row_flags.any() else 'Normal'
    for row_flags in flag_matrix
]

if not (sessions['anomaly_type_count'] == sessions['total_flags']).all():
    raise ValueError('anomaly_type_count phải bằng total_flags')
if not (sessions.loc[sessions['is_anomaly'] == 0, 'anomaly_types'] == 'Normal').all():
    raise ValueError('Session bình thường phải có anomaly_types = Normal')

rule_summary = (
    sessions[flag_cols]
    .sum()
    .rename('count')
    .reset_index()
    .rename(columns={'index': 'flag'})
)
rule_summary['anomaly_type'] = rule_summary['flag'].map(anomaly_type_map)
rule_summary['percent_sessions'] = rule_summary['count'] / len(sessions) * 100
rule_summary = rule_summary[['flag', 'anomaly_type', 'count', 'percent_sessions']].sort_values('count', ascending=False)
anomaly_type_breakdown = rule_summary[['anomaly_type', 'count', 'percent_sessions']].copy()

session_anomaly_tags = sessions[
    ['visitorid', 'session_number', 'is_anomaly', 'anomaly_types', 'anomaly_type_count', 'total_flags'] + flag_cols
].reset_index()

print('Ngưỡng luật nghiệp vụ ở cấp session:')
for threshold_name, threshold_value in THRESHOLDS.items():
    print(f'- {threshold_name}: {threshold_value}')

print(f'\nSố session bất thường theo luật: {sessions["is_anomaly_rule"].sum():,} / {len(sessions):,} '
      f'({sessions["is_anomaly_rule"].mean() * 100:.2f}%)')
display(rule_summary)
print('\nVí dụ multi-label anomaly tagging theo session:')
display(session_anomaly_tags[session_anomaly_tags['is_anomaly'] == 1].head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

plot_rules = rule_summary.sort_values('count', ascending=True)
axes[0].barh(plot_rules['anomaly_type'], plot_rules['count'], color=COLORS['warning'])
axes[0].set_title('Số phiên theo loại bất thường')
axes[0].set_xlabel('Số phiên')

flag_distribution = sessions['total_flags'].value_counts().sort_index()
axes[1].bar(flag_distribution.index.astype(str), flag_distribution.values, color=COLORS['primary'])
axes[1].set_title('Phân phối số loại bất thường mỗi phiên')
axes[1].set_xlabel('Số loại bất thường')
axes[1].set_ylabel('Số phiên')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_rule_breakdown.png', dpi=160, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
plot_type_breakdown = anomaly_type_breakdown.sort_values('count', ascending=True)
ax.barh(plot_type_breakdown['anomaly_type'], plot_type_breakdown['count'], color=COLORS['anomaly'])
ax.set_title('Phân rã loại bất thường đa nhãn theo phiên')
ax.set_xlabel('Số phiên')
ax.set_ylabel('Loại bất thường')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_anomaly_type_breakdown.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Kiểm tra rò rỉ nhãn theo luật nghiệp vụ

`full_feature_cols` chứa các feature tạo luật để minh họa rule-mimic. `safe_feature_cols` loại feature trực tiếp tạo nhãn và dùng cho bảng metric chính.

In [ ]:
leakage_audit_df = pd.DataFrame([
    {'rule': 'BR01_bot_scraper', 'business_condition': 'total_events >= 12 or events_per_minute >= 8 with enough events', 'rule_input_features': ['total_events', 'events_per_minute']},
    {'rule': 'BR02_ghost_buyer', 'business_condition': 'n_transaction > 0 and n_addtocart == 0', 'rule_input_features': ['n_transaction', 'n_addtocart']},
    {'rule': 'BR03_click_fraud', 'business_condition': 'n_view >= 6 and n_addtocart == 0 and n_transaction == 0', 'rule_input_features': ['n_view', 'n_addtocart', 'n_transaction']},
    {'rule': 'BR04_rapid_fire', 'business_condition': 'rapid_fire_count >= 1 or rapid_ratio >= 0.20 with enough events', 'rule_input_features': ['rapid_fire_count', 'rapid_ratio', 'total_events']},
    {'rule': 'BR05_night_crawler', 'business_condition': 'night_ratio >= 0.75 and total_events >= 4', 'rule_input_features': ['night_ratio', 'night_events', 'total_events']},
    {'rule': 'BR06_item_hoarding', 'business_condition': 'max_same_item_atc > 1', 'rule_input_features': ['max_same_item_atc']},
    {'rule': 'BR07_session_bomb', 'business_condition': 'unique_items >= 12', 'rule_input_features': ['unique_items']},
    {'rule': 'BR08_sequence_violation', 'business_condition': 'transaction before prior view/addtocart on same session-item timeline', 'rule_input_features': ['n_sequence_violation_events']},
    {'rule': 'BR09_transaction_burst', 'business_condition': 'n_transaction >= 3', 'rule_input_features': ['n_transaction']},
    {'rule': 'BR10_cart_abandonment', 'business_condition': 'n_addtocart >= 10 and n_transaction == 0', 'rule_input_features': ['n_addtocart', 'n_transaction']},
    {'rule': 'BR11_repeated_view_spam', 'business_condition': 'max_same_item_view >= 20', 'rule_input_features': ['max_same_item_view']},
    {'rule': 'BR12_category_scanning', 'business_condition': 'unique_categories >= 20', 'rule_input_features': ['unique_categories']},
])

full_feature_cols = [
    'total_events', 'unique_items', 'session_duration_sec', 'duration_min', 'events_per_minute',
    'n_view', 'n_addtocart', 'n_transaction', 'active_hours', 'unique_event_types',
    'event_type_entropy', 'hour_entropy', 'min_interval_sec', 'mean_interval_sec',
    'median_interval_sec', 'std_interval_sec', 'max_interval_sec', 'rapid_fire_count',
    'night_events', 'peak_events', 'weekend_events', 'view_rate', 'atc_rate', 'buy_rate',
    'night_ratio', 'peak_ratio', 'weekend_ratio', 'rapid_ratio', 'items_per_event',
    'view_to_cart_ratio', 'cart_to_transaction_ratio', 'max_same_item_view', 'max_same_item_atc',
    'unique_categories', 'unique_parent_categories', 'session_start_hour', 'session_dayofweek',
    'n_sequence_violation_events',
]

safe_feature_cols = [
    'session_duration_sec',
    'active_hours',
    'unique_event_types',
    'event_type_entropy',
    'hour_entropy',
    'mean_interval_sec',
    'median_interval_sec',
    'std_interval_sec',
    'max_interval_sec',
    'peak_events',
    'weekend_events',
    'peak_ratio',
    'weekend_ratio',
    'unique_parent_categories',
    'session_dayofweek',
]

leaky_feature_set = set(full_feature_cols).difference(safe_feature_cols)
leakage_audit_df['leaked_inputs_in_full_features'] = leakage_audit_df['rule_input_features'].apply(
    lambda cols: [col for col in cols if col in leaky_feature_set]
)
leakage_audit_df['inputs_remaining_in_safe_features'] = leakage_audit_df['rule_input_features'].apply(
    lambda cols: [col for col in cols if col in safe_feature_cols]
)
leakage_audit_df['n_leaked_inputs'] = leakage_audit_df['leaked_inputs_in_full_features'].str.len()

print('Kiểm tra rò rỉ nhãn: tập đặc trưng an toàn đã loại bỏ các đặc trưng trực tiếp tạo ra luật.')
display(leakage_audit_df)

if leakage_audit_df['inputs_remaining_in_safe_features'].str.len().sum() != 0:
    raise ValueError('safe_feature_cols vẫn chứa đặc trưng trực tiếp tạo nhãn luật')

fig, ax = plt.subplots(figsize=(12, 6))
plot_leakage = leakage_audit_df.sort_values('n_leaked_inputs', ascending=True)
ax.barh(plot_leakage['rule'], plot_leakage['n_leaked_inputs'], color=COLORS['anomaly'])
ax.set_title('Số đặc trưng tạo nhãn luật nằm trong tập đặc trưng đầy đủ')
ax.set_xlabel('Số đặc trưng có nguy cơ rò rỉ')
ax.set_ylabel('Luật nghiệp vụ')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_leakage_audit.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Chuan bi du lieu mo hinh

Target cua cac mo hinh giam sat la **ma tran 12 nhan** (`labels_multi`). Bang chinh dung **rule-aligned features** (`full_feature_cols`, gom ca dac trung lien quan truc tiep den luat) — muc tieu la do **muc do tai tao lai he thong business-rule**, nen chi so cao la hop le va co y nghia. Bang doi chung dung `safe_feature_cols` (da loai dac trung sinh luat) de cho thay phan nao do chinh xac den tu viec nhin thay dac trung sinh luat. Chia 60/20/20 tach nhom theo `visitorid`; chon nguong rieng tung nhan tren validation (co rang buoc ti le duong de tranh precision sup); test la ket qua bao cao.

In [ ]:
def build_clean_feature_matrix(source_dataframe, columns):
    feature_matrix = source_dataframe[columns].replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
    if feature_matrix.isna().any().any():
        raise ValueError('Ma tran dac trung van con NaN sau khi xu ly')
    if not np.isfinite(feature_matrix.to_numpy()).all():
        raise ValueError('Ma tran dac trung van con inf sau khi xu ly')
    return feature_matrix

features_full = build_clean_feature_matrix(sessions, full_feature_cols)  # = rule-aligned features (chua dac trung sinh luat)
features_safe = build_clean_feature_matrix(sessions, safe_feature_cols)

# Target nhi phan (giu lai de: chia/stratify, danh gia Isolation Forest, severity)
labels = sessions['is_anomaly_rule'].astype(int)
# Target da nhan: ma tran 12 cot, moi cot la mot anomaly type (multi-label)
labels_multi = sessions[flag_cols].astype(int)

if labels.nunique() != 2:
    raise ValueError('Nhan is_anomaly_rule phai co du 2 lop')

all_indices = np.arange(len(labels))
if MAX_SUPERVISED_ROWS is not None and len(labels) > MAX_SUPERVISED_ROWS:
    supervised_indices, _ = train_test_split(
        all_indices,
        train_size=MAX_SUPERVISED_ROWS,
        random_state=RANDOM_STATE,
        stratify=labels,
    )
else:
    supervised_indices = all_indices

visitor_ids_all = sessions['visitorid'].to_numpy()
visitor_ids_supervised = visitor_ids_all[supervised_indices]
labels_supervised = labels.iloc[supervised_indices].to_numpy()

group_splitter_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_validation_positions, test_positions = next(group_splitter_test.split(supervised_indices, labels_supervised, visitor_ids_supervised))
train_validation_indices = supervised_indices[train_validation_positions]
test_indices = supervised_indices[test_positions]

visitor_ids_train_validation = visitor_ids_all[train_validation_indices]
labels_train_validation = labels.iloc[train_validation_indices].to_numpy()
group_splitter_validation = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_positions, validation_positions = next(group_splitter_validation.split(train_validation_indices, labels_train_validation, visitor_ids_train_validation))
train_indices = train_validation_indices[train_positions]
validation_indices = train_validation_indices[validation_positions]

features_safe_train = features_safe.iloc[train_indices]
features_safe_validation = features_safe.iloc[validation_indices]
features_safe_test = features_safe.iloc[test_indices]
features_full_train = features_full.iloc[train_indices]
features_full_validation = features_full.iloc[validation_indices]
features_full_test = features_full.iloc[test_indices]

# Nhan nhi phan theo split (cho Isolation Forest va tham chieu)
labels_train = labels.iloc[train_indices]
labels_validation = labels.iloc[validation_indices]
labels_test = labels.iloc[test_indices]

# Nhan da nhan theo split
labels_multi_train = labels_multi.iloc[train_indices]
labels_multi_validation = labels_multi.iloc[validation_indices]
labels_multi_test = labels_multi.iloc[test_indices]

# Chi mo hinh hoa cac nhan co du 2 lop trong train (tranh loi single-class)
MIN_LABEL_POSITIVES = 30  # nhan qua hiem se cho threshold sup -> bo qua, danh gia khong dang tin
model_label_cols = [
    c for c in flag_cols
    if labels_multi_train[c].nunique() == 2 and int(labels_multi_train[c].sum()) >= MIN_LABEL_POSITIVES
]
dropped_label_cols = [c for c in flag_cols if c not in model_label_cols]
model_label_names = [anomaly_type_map[c] for c in model_label_cols]

labels_multi_train_model = labels_multi_train[model_label_cols]
labels_multi_validation_model = labels_multi_validation[model_label_cols]
labels_multi_test_model = labels_multi_test[model_label_cols]
labels_multi_all_model = labels_multi[model_label_cols]

safe_feature_scaler = RobustScaler()
safe_feature_scaler.fit(features_safe_train)
features_safe_scaled = safe_feature_scaler.transform(features_safe).astype('float32')
features_safe_scaled_train = features_safe_scaled[train_indices]
features_safe_scaled_validation = features_safe_scaled[validation_indices]
features_safe_scaled_test = features_safe_scaled[test_indices]

full_feature_scaler = RobustScaler()
full_feature_scaler.fit(features_full_train)
features_full_scaled = full_feature_scaler.transform(features_full).astype('float32')

print(f'Toan bo session profile: {len(features_safe):,} dong')
print(f'Mau supervised train/validation/test: {len(supervised_indices):,} dong')
print(f'Train: {len(train_indices):,}; Validation: {len(validation_indices):,}; Test: {len(test_indices):,}')
print(f'Ti le anomaly (nhi phan) trong toan bo session: {labels.mean() * 100:.2f}%')
print(f'So nhan duoc mo hinh hoa: {len(model_label_cols)}/{len(flag_cols)}')
if dropped_label_cols:
    print(f'Nhan bi bo (thieu lop hoac < {MIN_LABEL_POSITIVES} mau duong trong train): {[anomaly_type_map[c] for c in dropped_label_cols]}')

print('\nSo session duong tinh moi nhan trong train (multi-label):')
label_support_train = labels_multi_train.sum().rename('train_positive')
label_support_test = labels_multi_test.sum().rename('test_positive')
label_support = pd.concat([label_support_train, label_support_test], axis=1)
label_support.index = [anomaly_type_map[c] for c in label_support.index]
display(label_support)

visitors_train = set(visitor_ids_all[train_indices])
visitors_validation = set(visitor_ids_all[validation_indices])
visitors_test = set(visitor_ids_all[test_indices])
overlap_train_validation = visitors_train & visitors_validation
overlap_train_test = visitors_train & visitors_test
overlap_validation_test = visitors_validation & visitors_test
print(f'Visitor train/validation/test: {len(visitors_train):,} / {len(visitors_validation):,} / {len(visitors_test):,}')
print(f'Trung visitor train-validation: {len(overlap_train_validation)}; train-test: {len(overlap_train_test)}; validation-test: {len(overlap_validation_test)}')
assert not (overlap_train_validation or overlap_train_test or overlap_validation_test), 'Van con visitor bi ro ri giua cac tap!'
print('OK: khong co visitor nao bi ro ri giua train/validation/test (tach nhom theo visitorid).')


In [ ]:
BASELINE_EVALUATION = 'Baseline (multi-label)'
MAIN_SUPERVISED_EVALUATION = 'Tai tao luat nghiep vu - rule-aligned features (multi-label)'
LEAKAGE_CONTROL_EVALUATION = 'Doi chung giam leakage - safe features (multi-label)'
UNSUPERVISED_EVALUATION = 'Unsupervised overlap voi nhan gia phien (nhi phan)'
SANITY_CHECK_EVALUATION = 'Sanity check: shuffle nhan thi phai te di'

from sklearn.metrics import average_precision_score

evaluation_records = []
per_label_records = []
confusion_prediction_sets = {}


def proba_matrix(model, features, n_labels):
    probabilities = model.predict_proba(features)
    if not isinstance(probabilities, list):
        probabilities = [probabilities]
    columns = []
    for proba in probabilities:
        proba = np.asarray(proba)
        if proba.ndim == 2 and proba.shape[1] == 2:
            columns.append(proba[:, 1])
        elif proba.ndim == 2 and proba.shape[1] == 1:
            columns.append(proba[:, 0])
        else:
            columns.append(np.zeros(proba.shape[0]))
    score_matrix = np.column_stack(columns)
    if score_matrix.shape[1] != n_labels:
        raise ValueError('So cot xac suat khong khop so nhan')
    return score_matrix


def choose_thresholds_per_label(true_matrix, score_matrix, label, max_pos_rate_multiple=3.0):
    """Chon nguong toi da F1 rieng cho tung nhan tren validation, NHUNG chi xet cac nguong
    cho ti le du doan duong <= max_pos_rate_multiple lan ti le that. Rang buoc nay chan
    hien tuong 'recall=1 / precision=0' tren nhan mat can bang nang."""
    true_matrix = np.asarray(true_matrix).astype(int)
    score_matrix = np.asarray(score_matrix, dtype='float64')
    n_labels = score_matrix.shape[1]
    thresholds = np.full(n_labels, 0.5)
    for j in range(n_labels):
        scores = score_matrix[:, j]
        truths = true_matrix[:, j]
        if len(np.unique(scores)) <= 1 or len(np.unique(truths)) <= 1:
            thresholds[j] = 0.5
            continue
        base_rate = float(truths.mean())
        cap = min(1.0, max(base_rate * max_pos_rate_multiple, base_rate + 0.005))
        candidates = np.unique(np.r_[0.5, np.quantile(scores, np.linspace(0.01, 0.999, 99))])
        best_threshold, best_f1 = None, -1.0
        for t in candidates:
            predictions = (scores >= t).astype(int)
            if predictions.mean() > cap:   # chan over-predict
                continue
            value = f1_score(truths, predictions, zero_division=0)
            if value > best_f1:
                best_f1, best_threshold = value, float(t)
        if best_threshold is None:        # tat ca deu vuot cap -> chon nguong cao nhat
            best_threshold = float(candidates.max())
        thresholds[j] = best_threshold
    print(f'{label}: da chon nguong rieng cho {n_labels} nhan (co rang buoc ti le duong)')
    return thresholds


def predict_with_thresholds(score_matrix, thresholds):
    return (np.asarray(score_matrix) >= np.asarray(thresholds)).astype(int)


def record_multilabel(model_name, feature_set, evaluation_type, split,
                      true_matrix, prediction_matrix, label_names,
                      score_matrix=None, store_confusion=True):
    true_matrix = np.asarray(true_matrix).astype(int)
    prediction_matrix = np.asarray(prediction_matrix).astype(int)

    macro_f1 = f1_score(true_matrix, prediction_matrix, average='macro', zero_division=0)
    micro_f1 = f1_score(true_matrix, prediction_matrix, average='micro', zero_division=0)
    weighted_f1 = f1_score(true_matrix, prediction_matrix, average='weighted', zero_division=0)
    macro_precision = precision_score(true_matrix, prediction_matrix, average='macro', zero_division=0)
    macro_recall = recall_score(true_matrix, prediction_matrix, average='macro', zero_division=0)
    micro_precision = precision_score(true_matrix, prediction_matrix, average='micro', zero_division=0)
    micro_recall = recall_score(true_matrix, prediction_matrix, average='micro', zero_division=0)
    hamming = hamming_loss(true_matrix, prediction_matrix)
    subset_accuracy = accuracy_score(true_matrix, prediction_matrix)

    macro_pr_auc = np.nan
    per_label_pr_auc = [np.nan] * len(label_names)
    if score_matrix is not None:
        score_matrix = np.asarray(score_matrix, dtype='float64')
        aps = []
        for j in range(len(label_names)):
            if len(np.unique(true_matrix[:, j])) == 2:
                ap = average_precision_score(true_matrix[:, j], score_matrix[:, j])
                per_label_pr_auc[j] = ap
                aps.append(ap)
        macro_pr_auc = float(np.mean(aps)) if aps else np.nan

    evaluation_records.append({
        'model': model_name, 'feature_set': feature_set, 'evaluation_type': evaluation_type, 'split': split,
        'macro_f1': macro_f1, 'micro_f1': micro_f1, 'weighted_f1': weighted_f1,
        'macro_precision': macro_precision, 'macro_recall': macro_recall,
        'micro_precision': micro_precision, 'micro_recall': micro_recall,
        'macro_pr_auc': macro_pr_auc, 'hamming_loss': hamming, 'subset_accuracy': subset_accuracy,
        'n_labels': len(label_names), 'support_total': int(true_matrix.sum()),
        'predicted_total': int(prediction_matrix.sum()),
    })

    precision, recall, f1_values, support = precision_recall_fscore_support(
        true_matrix, prediction_matrix, average=None, zero_division=0)
    for j, name in enumerate(label_names):
        per_label_records.append({
            'model': model_name, 'feature_set': feature_set, 'split': split, 'label': name,
            'precision': precision[j], 'recall': recall[j], 'f1_score': f1_values[j],
            'pr_auc': per_label_pr_auc[j], 'support': int(support[j]),
        })

    if store_confusion:
        confusion_prediction_sets[f'{model_name} ({feature_set})'] = {
            'y_true_any': (true_matrix.max(axis=1) > 0).astype(int),
            'y_pred_any': (prediction_matrix.max(axis=1) > 0).astype(int),
            'split': split,
        }

    print()
    print(f'{model_name} - {feature_set} - {evaluation_type} - {split}')
    print(f'macro-F1={macro_f1:.4f} | micro-F1={micro_f1:.4f} | macro-PR-AUC={macro_pr_auc:.4f} | '
          f'Hamming={hamming:.4f} | subset-acc={subset_accuracy:.4f}')
    print(classification_report(true_matrix, prediction_matrix, target_names=label_names, zero_division=0))


def record_binary_overlap(model_name, feature_set, evaluation_type, split,
                          true_binary, prediction_binary, score_binary=None, store_confusion=True):
    true_binary = np.asarray(true_binary).astype(int)
    prediction_binary = np.asarray(prediction_binary).astype(int)
    f1_value = f1_score(true_binary, prediction_binary, zero_division=0)
    precision_value = precision_score(true_binary, prediction_binary, zero_division=0)
    recall_value = recall_score(true_binary, prediction_binary, zero_division=0)
    accuracy_value = accuracy_score(true_binary, prediction_binary)
    pr_auc = average_precision_score(true_binary, score_binary) if score_binary is not None and len(np.unique(true_binary)) == 2 else np.nan

    evaluation_records.append({
        'model': model_name, 'feature_set': feature_set, 'evaluation_type': evaluation_type, 'split': split,
        'macro_f1': f1_value, 'micro_f1': f1_value, 'weighted_f1': f1_value,
        'macro_precision': precision_value, 'macro_recall': recall_value,
        'micro_precision': precision_value, 'micro_recall': recall_value,
        'macro_pr_auc': pr_auc, 'hamming_loss': 1 - accuracy_value, 'subset_accuracy': accuracy_value,
        'n_labels': 1, 'support_total': int(true_binary.sum()), 'predicted_total': int(prediction_binary.sum()),
    })
    per_label_records.append({
        'model': model_name, 'feature_set': feature_set, 'split': split, 'label': '(overlap nhi phan)',
        'precision': precision_value, 'recall': recall_value, 'f1_score': f1_value,
        'pr_auc': pr_auc, 'support': int(true_binary.sum()),
    })
    if store_confusion:
        confusion_prediction_sets[f'{model_name} ({feature_set})'] = {
            'y_true_any': true_binary, 'y_pred_any': prediction_binary, 'split': split}
    print()
    print(f'{model_name} - {feature_set} - {evaluation_type} - {split}')
    print(classification_report(true_binary, prediction_binary, target_names=['Normal', 'Anomaly'], zero_division=0))


def train_and_evaluate_multilabel(model, model_name,
                                  features_train, labels_train_matrix,
                                  features_validation, labels_validation_matrix,
                                  features_test, labels_test_matrix,
                                  features_all, label_names,
                                  feature_set='rule_aligned_features',
                                  evaluation_type=MAIN_SUPERVISED_EVALUATION,
                                  store_test_confusion=True):
    n_labels = len(label_names)
    model.fit(features_train, labels_train_matrix)

    validation_scores = proba_matrix(model, features_validation, n_labels)
    thresholds = choose_thresholds_per_label(
        labels_validation_matrix.to_numpy(), validation_scores, f'{model_name} ({feature_set})')

    validation_predictions = predict_with_thresholds(validation_scores, thresholds)
    record_multilabel(model_name, feature_set, evaluation_type, 'validation',
                      labels_validation_matrix.to_numpy(), validation_predictions,
                      label_names, score_matrix=validation_scores, store_confusion=False)

    test_scores = proba_matrix(model, features_test, n_labels)
    test_predictions = predict_with_thresholds(test_scores, thresholds)
    record_multilabel(model_name, feature_set, evaluation_type, 'test',
                      labels_test_matrix.to_numpy(), test_predictions,
                      label_names, score_matrix=test_scores, store_confusion=store_test_confusion)

    all_scores = proba_matrix(model, features_all, n_labels)
    all_predictions = predict_with_thresholds(all_scores, thresholds)
    return model, thresholds, all_scores, all_predictions


def predicted_types_string(prediction_matrix, label_names):
    label_names = np.array(label_names, dtype=object)
    output = []
    for row in np.asarray(prediction_matrix).astype(bool):
        output.append(', '.join(label_names[row].tolist()) if row.any() else 'Normal')
    return output


## 7. Baseline va mo hinh supervised (multi-label, rule-aligned features)

Bon mo hinh du doan dong thoi 12 anomaly type tren rule-aligned features. Decision Tree & Random Forest multi-output truc tiep; XGBoost & LightGBM qua `MultiOutputClassifier`. Muc tieu: tai tao lai he thong business-rule -> tat ca chi so (precision/recall/F1/PR-AUC) deu cao.

In [ ]:
dummy_model = MultiOutputClassifier(
    DummyClassifier(strategy='stratified', random_state=RANDOM_STATE), n_jobs=-1)
dummy_model.fit(features_full_train, labels_multi_train_model)

dummy_validation_scores = proba_matrix(dummy_model, features_full_validation, len(model_label_cols))
dummy_validation_predictions = predict_with_thresholds(dummy_validation_scores, np.full(len(model_label_cols), 0.5))
record_multilabel('Dummy baseline', 'rule_aligned_features', BASELINE_EVALUATION, 'validation',
                  labels_multi_validation_model.to_numpy(), dummy_validation_predictions,
                  model_label_names, score_matrix=dummy_validation_scores, store_confusion=False)

dummy_test_scores = proba_matrix(dummy_model, features_full_test, len(model_label_cols))
dummy_test_predictions = predict_with_thresholds(dummy_test_scores, np.full(len(model_label_cols), 0.5))
record_multilabel('Dummy baseline', 'rule_aligned_features', BASELINE_EVALUATION, 'test',
                  labels_multi_test_model.to_numpy(), dummy_test_predictions,
                  model_label_names, score_matrix=dummy_test_scores, store_confusion=False)


<a id="section-xgboost"></a>

### Dinh Tuan - XGBoost (multi-label)

In [ ]:
def make_xgboost_model(n_estimators=200):
    return XGBClassifier(
        n_estimators=n_estimators, max_depth=6, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9, objective='binary:logistic',
        eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1,
    )

xgboost_model = MultiOutputClassifier(make_xgboost_model(n_estimators=200), n_jobs=-1)
(
    xgboost_model,
    xgboost_thresholds,
    xgboost_scores_all,
    xgboost_predictions_all,
) = train_and_evaluate_multilabel(
    model=xgboost_model, model_name='XGBoost',
    features_train=features_full_train, labels_train_matrix=labels_multi_train_model,
    features_validation=features_full_validation, labels_validation_matrix=labels_multi_validation_model,
    features_test=features_full_test, labels_test_matrix=labels_multi_test_model,
    features_all=features_full, label_names=model_label_names,
    feature_set='rule_aligned_features', evaluation_type=MAIN_SUPERVISED_EVALUATION,
)
sessions['xgboost_pred'] = xgboost_predictions_all.max(axis=1)
sessions['xgboost_n_pred_types'] = xgboost_predictions_all.sum(axis=1)
sessions['xgboost_score'] = xgboost_scores_all.max(axis=1)
sessions['xgboost_pred_types'] = predicted_types_string(xgboost_predictions_all, model_label_names)


### Le Van Anh - Decision Tree (multi-output truc tiep)

In [ ]:
# Mot cay cho moi nhan (one-vs-rest): moi cay tai tao tron ven luat cua nhan do.
decision_tree_model = MultiOutputClassifier(
    DecisionTreeClassifier(max_depth=None, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE), n_jobs=-1)
(
    decision_tree_model,
    decision_tree_thresholds,
    decision_tree_scores_all,
    decision_tree_predictions_all,
) = train_and_evaluate_multilabel(
    model=decision_tree_model, model_name='Decision Tree',
    features_train=features_full_train, labels_train_matrix=labels_multi_train_model,
    features_validation=features_full_validation, labels_validation_matrix=labels_multi_validation_model,
    features_test=features_full_test, labels_test_matrix=labels_multi_test_model,
    features_all=features_full, label_names=model_label_names,
    feature_set='rule_aligned_features', evaluation_type=MAIN_SUPERVISED_EVALUATION,
)
sessions['decision_tree_pred'] = decision_tree_predictions_all.max(axis=1)
sessions['decision_tree_n_pred_types'] = decision_tree_predictions_all.sum(axis=1)
sessions['decision_tree_score'] = decision_tree_scores_all.max(axis=1)
sessions['decision_tree_pred_types'] = predicted_types_string(decision_tree_predictions_all, model_label_names)

# Cay multi-output value 12 chieu kho doc -> ve cay phu DON NHAN (is_anomaly) chi de MINH HOA luat quyet dinh.
illustration_tree = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=100, class_weight='balanced', random_state=RANDOM_STATE)
illustration_tree.fit(features_full_train, labels_train)
fig, ax = plt.subplots(figsize=(24, 12))
plot_tree(illustration_tree, feature_names=full_feature_cols, class_names=['Binh thuong', 'Bat thuong'],
          filled=True, rounded=True, max_depth=3, fontsize=8, ax=ax)
ax.set_title('MINH HOA: cay quyet dinh don nhan (is_anomaly) tren rule-aligned features')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_decision_tree.png', dpi=160, bbox_inches='tight')
plt.show()


<a id="section-random-forest"></a>

### Tuan Anh - Random Forest (multi-output truc tiep)

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=5,
    class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
(
    random_forest_model,
    random_forest_thresholds,
    random_forest_scores_all,
    random_forest_predictions_all,
) = train_and_evaluate_multilabel(
    model=random_forest_model, model_name='Random Forest',
    features_train=features_full_train, labels_train_matrix=labels_multi_train_model,
    features_validation=features_full_validation, labels_validation_matrix=labels_multi_validation_model,
    features_test=features_full_test, labels_test_matrix=labels_multi_test_model,
    features_all=features_full, label_names=model_label_names,
    feature_set='rule_aligned_features', evaluation_type=MAIN_SUPERVISED_EVALUATION,
)
sessions['random_forest_pred'] = random_forest_predictions_all.max(axis=1)
sessions['random_forest_n_pred_types'] = random_forest_predictions_all.sum(axis=1)
sessions['random_forest_score'] = random_forest_scores_all.max(axis=1)
sessions['random_forest_pred_types'] = predicted_types_string(random_forest_predictions_all, model_label_names)


### Thuy - LightGBM (multi-label)

In [ ]:
def make_lightgbm_model(n_estimators=300):
    return LGBMClassifier(
        n_estimators=n_estimators, learning_rate=0.1, num_leaves=63, max_depth=-1,
        min_child_samples=30, subsample=0.9, colsample_bytree=0.9, objective='binary',
        random_state=RANDOM_STATE, n_jobs=-1, reg_lambda=1.0, verbosity=-1, force_col_wise=True)

lightgbm_model = MultiOutputClassifier(make_lightgbm_model(n_estimators=300), n_jobs=-1)
(
    lightgbm_model,
    lightgbm_thresholds,
    lightgbm_scores_all,
    lightgbm_predictions_all,
) = train_and_evaluate_multilabel(
    model=lightgbm_model, model_name='LightGBM',
    features_train=features_full_train, labels_train_matrix=labels_multi_train_model,
    features_validation=features_full_validation, labels_validation_matrix=labels_multi_validation_model,
    features_test=features_full_test, labels_test_matrix=labels_multi_test_model,
    features_all=features_full, label_names=model_label_names,
    feature_set='rule_aligned_features', evaluation_type=MAIN_SUPERVISED_EVALUATION,
)
sessions['lightgbm_pred'] = lightgbm_predictions_all.max(axis=1)
sessions['lightgbm_n_pred_types'] = lightgbm_predictions_all.sum(axis=1)
sessions['lightgbm_score'] = lightgbm_scores_all.max(axis=1)
sessions['lightgbm_pred_types'] = predicted_types_string(lightgbm_predictions_all, model_label_names)


## 8. Doi chung trung thuc: safe features (giam leakage)

Huan luyen lai 4 mo hinh tren `safe_feature_cols`. Chi so thap hon han bang chinh — day chinh la bang chung khoa hoc rang phan lon do chinh xac cao o bang chinh den tu label leakage (mo hinh nhin thay dac trung sinh luat), khong phai nang luc phat hien gian lan that.

In [ ]:
# Doi chung TRUNG THUC: train lai 4 mo hinh tren SAFE features (da loai dac trung sinh luat).
# Ket qua thap hon han bang chinh -> cho thay phan lon do chinh xac cao o bang chinh den tu viec
# mo hinh "nhin thay" dac trung sinh luat (rule reproduction), KHONG phai phat hien gian lan that.
for _name, _ctor in [
    ('XGBoost', lambda: MultiOutputClassifier(make_xgboost_model(n_estimators=200), n_jobs=-1)),
    ('LightGBM', lambda: MultiOutputClassifier(make_lightgbm_model(n_estimators=300), n_jobs=-1)),
    ('Decision Tree', lambda: DecisionTreeClassifier(max_depth=16, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_STATE)),
    ('Random Forest', lambda: RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=5, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)),
]:
    train_and_evaluate_multilabel(
        model=_ctor(), model_name=_name,
        features_train=features_safe_train, labels_train_matrix=labels_multi_train_model,
        features_validation=features_safe_validation, labels_validation_matrix=labels_multi_validation_model,
        features_test=features_safe_test, labels_test_matrix=labels_multi_test_model,
        features_all=features_safe, label_names=model_label_names,
        feature_set='safe_features', evaluation_type=LEAKAGE_CONTROL_EVALUATION, store_test_confusion=False,
    )


## 9. Shuffle-label sanity check (multi-label)

Trao ngau nhien thu tu dong cua ma tran nhan train. Neu pipeline khong con loi nghiem trong, macro-F1 tren test phai sup ve gan muc nhieu.

In [ ]:
# Shuffle nhan train tren rule-aligned features. Du co dac trung sinh luat, neu tron nhan thi
# lien ket dac trung<->nhan bi pha -> macro-F1 test phai sup. Kiem tra pipeline khong "an gian".
rng = np.random.default_rng(RANDOM_STATE)
shuffled_rows = rng.permutation(len(labels_multi_train_model))
shuffled_labels_multi_train = pd.DataFrame(
    labels_multi_train_model.to_numpy()[shuffled_rows],
    columns=model_label_cols, index=labels_multi_train_model.index)

xgboost_shuffle_model = MultiOutputClassifier(make_xgboost_model(n_estimators=80), n_jobs=-1)
xgboost_shuffle_model.fit(features_full_train, shuffled_labels_multi_train)

shuffle_validation_scores = proba_matrix(xgboost_shuffle_model, features_full_validation, len(model_label_cols))
shuffle_thresholds = choose_thresholds_per_label(
    labels_multi_validation_model.to_numpy(), shuffle_validation_scores, 'XGBoost shuffled-label sanity check')
shuffle_validation_predictions = predict_with_thresholds(shuffle_validation_scores, shuffle_thresholds)
record_multilabel('XGBoost shuffled-label sanity check', 'rule_aligned_features', SANITY_CHECK_EVALUATION,
                  'validation', labels_multi_validation_model.to_numpy(), shuffle_validation_predictions,
                  model_label_names, score_matrix=shuffle_validation_scores, store_confusion=False)

shuffle_test_scores = proba_matrix(xgboost_shuffle_model, features_full_test, len(model_label_cols))
shuffle_test_predictions = predict_with_thresholds(shuffle_test_scores, shuffle_thresholds)
record_multilabel('XGBoost shuffled-label sanity check', 'rule_aligned_features', SANITY_CHECK_EVALUATION,
                  'test', labels_multi_test_model.to_numpy(), shuffle_test_predictions,
                  model_label_names, score_matrix=shuffle_test_scores, store_confusion=False)

shuffle_macro_f1 = f1_score(labels_multi_test_model.to_numpy(), shuffle_test_predictions, average='macro', zero_division=0)
print(f'\nmacro-F1 shuffled-label tren test: {shuffle_macro_f1:.4f}')
print('Sanity check dat: mo hinh khong con doan duoc khi nhan bi trao ngau nhien.'
      if shuffle_macro_f1 < 0.30 else 'Canh bao: macro-F1 shuffled-label cao bat thuong, kiem tra lai pipeline.')


## 10. Feature importance tren safe session feature set

Voi mo hinh multi-output (DT, RF) lay `feature_importances_` co san; voi `MultiOutputClassifier` (XGBoost, LightGBM) lay trung binh do quan trong qua 12 estimator con.

In [ ]:
def get_feature_importances(model):
    if hasattr(model, 'feature_importances_'):
        return np.asarray(model.feature_importances_, dtype='float64')
    if hasattr(model, 'estimators_'):
        importances = [np.asarray(est.feature_importances_, dtype='float64')
                       for est in model.estimators_ if hasattr(est, 'feature_importances_')]
        if importances:
            return np.mean(importances, axis=0)
    return None

def top_feature_importances(model, feature_names, model_name, top_n=12):
    importances = get_feature_importances(model)
    if importances is None:
        return pd.DataFrame(columns=['feature', 'importance', 'model'])
    frame = pd.DataFrame({'feature': feature_names, 'importance': importances, 'model': model_name})
    return frame.sort_values('importance', ascending=False).head(top_n)

importance_models = [
    ('XGBoost', xgboost_model, COLORS['secondary']),
    ('LightGBM', lightgbm_model, COLORS['warning']),
    ('Random Forest', random_forest_model, COLORS['primary']),
]
feature_importances = pd.concat(
    [top_feature_importances(model, full_feature_cols, model_name) for model_name, model, _ in importance_models],
    ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, (model_name, _, color) in zip(axes, importance_models):
    subset = feature_importances[feature_importances['model'] == model_name].sort_values('importance', ascending=True)
    ax.barh(subset['feature'], subset['importance'], color=color)
    ax.set_title(f'Do quan trong dac trung (trung binh 12 nhan) - {model_name}')
    ax.set_xlabel('Do quan trong')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()
display(feature_importances)


<a id="section-isolation-forest"></a>

## 11. Đức Anh - Isolation Forest

Isolation Forest fit trên train không cần nhãn. Validation dùng để chọn ngưỡng anomaly score, sau đó test dùng để báo cáo overlap với pseudo-label.


In [ ]:
# Isolation Forest van la KHONG GIAM SAT: fit khong dung nhan, chi ra MOT anomaly score
# -> khong multi-label duoc. Danh gia bang overlap nhi phan voi is_anomaly_rule.
# Luu y: nguong duoc chon tren nhan validation, nen operating point co dung nhan.
if IFOREST_TRAIN_ROWS is not None and len(train_indices) > IFOREST_TRAIN_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    isolation_forest_train_indices = rng.choice(train_indices, size=IFOREST_TRAIN_ROWS, replace=False)
else:
    isolation_forest_train_indices = train_indices

isolation_forest_model = IsolationForest(n_estimators=150, contamination='auto', random_state=RANDOM_STATE, n_jobs=-1)
isolation_forest_model.fit(features_safe_scaled[isolation_forest_train_indices])

isolation_forest_score = -isolation_forest_model.score_samples(features_safe_scaled)


def choose_binary_threshold(true_binary, scores):
    true_binary = np.asarray(true_binary).astype(int)
    scores = np.asarray(scores, dtype='float64')
    if len(np.unique(scores)) <= 1 or len(np.unique(true_binary)) <= 1:
        return float(np.quantile(scores, 0.95))
    candidates = np.unique(np.r_[np.quantile(scores, np.linspace(0.50, 0.999, 80))])
    best_threshold, best_f1 = candidates[0], -1.0
    for threshold in candidates:
        predictions = (scores >= threshold).astype(int)
        value = f1_score(true_binary, predictions, zero_division=0)
        if value > best_f1:
            best_f1, best_threshold = value, float(threshold)
    return best_threshold

isolation_forest_threshold = choose_binary_threshold(
    labels_validation.to_numpy(), isolation_forest_score[validation_indices])
isolation_forest_prediction = (isolation_forest_score >= isolation_forest_threshold).astype(int)

sessions['isolation_forest_score'] = isolation_forest_score
sessions['isolation_forest_pred'] = isolation_forest_prediction

record_binary_overlap('Isolation Forest', 'safe_features', UNSUPERVISED_EVALUATION, 'validation',
                      labels_validation.to_numpy(), isolation_forest_prediction[validation_indices],
                      score_binary=isolation_forest_score[validation_indices], store_confusion=False)
record_binary_overlap('Isolation Forest', 'safe_features', UNSUPERVISED_EVALUATION, 'test',
                      labels_test.to_numpy(), isolation_forest_prediction[test_indices],
                      score_binary=isolation_forest_score[test_indices])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(sessions['isolation_forest_score'], bins=60, color=COLORS['primary'], ax=axes[0])
axes[0].axvline(isolation_forest_threshold, color=COLORS['anomaly'], linestyle='--')
axes[0].set_title('Phan phoi diem bat thuong Isolation Forest')
axes[0].set_xlabel('Diem cang cao = cang bat thuong')

plot_sample = sessions.sample(min(PLOT_SAMPLE_ROWS, len(sessions)), random_state=RANDOM_STATE)
sns.scatterplot(data=plot_sample, x='session_duration_sec', y='isolation_forest_score', hue='isolation_forest_pred', palette={0: COLORS['normal'], 1: COLORS['anomaly']}, alpha=0.55, linewidth=0, s=18, ax=axes[1])
axes[1].set_title('Diem so theo thoi luong phien')
axes[1].set_xlabel('Thoi luong phien (giay)')
axes[1].set_ylabel('Diem Isolation Forest')
axes[1].legend(title='Bat thuong (IF)')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_isolation_scores.png', dpi=160, bbox_inches='tight')
plt.show()


## 12. So sanh cac thuat toan (multi-label)

Bang chinh = rule-aligned features (tai tao luat, chi so cao). Bang doi chung = safe features (trung thuc, thap hon). Bao cao macro/micro-F1, **PR-AUC** (khong phu thuoc nguong) va Hamming loss. Isolation Forest chi co overlap nhi phan.

In [ ]:
model_metrics = pd.DataFrame(evaluation_records)
metric_columns = ['macro_f1', 'micro_f1', 'weighted_f1', 'macro_precision', 'macro_recall',
                  'micro_precision', 'micro_recall', 'macro_pr_auc', 'hamming_loss', 'subset_accuracy']
model_metrics[metric_columns] = model_metrics[metric_columns].round(4)

main_metrics = model_metrics[model_metrics['evaluation_type'].isin(
    [BASELINE_EVALUATION, MAIN_SUPERVISED_EVALUATION, UNSUPERVISED_EVALUATION])].copy()
main_test_metrics = main_metrics[main_metrics['split'] == 'test'].copy()
leakage_control_metrics = model_metrics[model_metrics['evaluation_type'] == LEAKAGE_CONTROL_EVALUATION].copy()

print('BANG CHINH - rule-aligned features (multi-label): macro/micro F1, PR-AUC')
display(main_metrics.sort_values(['split', 'model'])[
    ['model', 'evaluation_type', 'split', 'macro_f1', 'micro_f1', 'macro_pr_auc',
     'macro_precision', 'macro_recall', 'hamming_loss', 'subset_accuracy', 'n_labels']])
print('Bang test (bao cao cuoi):')
display(main_test_metrics[['model', 'macro_f1', 'micro_f1', 'macro_pr_auc',
                           'macro_precision', 'macro_recall', 'hamming_loss', 'subset_accuracy']])
print('DOI CHUNG TRUNG THUC - safe features (giam leakage): so thap hon = bang chung leakage o bang chinh')
display(leakage_control_metrics.sort_values(['split', 'model'])[
    ['model', 'split', 'macro_f1', 'micro_f1', 'macro_pr_auc', 'hamming_loss']])

fig, ax = plt.subplots(figsize=(13, 6))
plot_metrics = main_test_metrics.set_index('model')[['macro_f1', 'micro_f1', 'macro_pr_auc', 'macro_precision', 'macro_recall']]
plot_metrics.plot(kind='bar', ax=ax,
                  color=[COLORS['primary'], COLORS['warning'], COLORS['secondary'], COLORS['muted'], COLORS['normal']])
ax.set_title('So sanh chi so test (rule-aligned features, multi-label)')
ax.set_xlabel('Thuat toan'); ax.set_ylabel('Gia tri'); ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=25); ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

per_label_df = pd.DataFrame(per_label_records)
supervised_model_order = ['XGBoost', 'Decision Tree', 'Random Forest', 'LightGBM']
per_label_test = per_label_df[
    (per_label_df['split'] == 'test') & (per_label_df['feature_set'] == 'rule_aligned_features')
    & (per_label_df['model'].isin(supervised_model_order)) & (per_label_df['label'] != '(overlap nhi phan)')].copy()
if not per_label_test.empty:
    f1_pivot = per_label_test.pivot_table(index='label', columns='model', values='f1_score', aggfunc='mean')
    f1_pivot = f1_pivot.reindex(columns=[m for m in supervised_model_order if m in f1_pivot.columns])
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(f1_pivot, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1,
                cbar_kws={'label': 'F1-score'}, ax=ax)
    ax.set_title('F1 theo tung anomaly type (test, rule-aligned features)')
    ax.set_xlabel('Thuat toan'); ax.set_ylabel('Anomaly type')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'output_group_per_label_f1.png', dpi=160, bbox_inches='tight')
    plt.show()

model_order = ['XGBoost', 'Decision Tree', 'Random Forest', 'LightGBM', 'Isolation Forest']
vt_plot = main_metrics[main_metrics['model'].isin(model_order)].copy()
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=vt_plot, x='model', y='macro_f1', hue='split', order=model_order,
            palette=[COLORS['muted'], COLORS['primary']], ax=ax)
ax.set_title('macro-F1: validation vs test (rule-aligned features)')
ax.set_xlabel('Thuat toan'); ax.set_ylabel('macro-F1'); ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_validation_test_f1.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Multi-label khong co mot ma tran nham lan duy nhat. De truc quan, ta xem
# bai toan rut gon "co bat thuong nao khong" (any-label) cua tung mo hinh.
confusion_items = {
    key: value
    for key, value in confusion_prediction_sets.items()
    if key in [
        'XGBoost (safe_features)',
        'Decision Tree (safe_features)',
        'Random Forest (safe_features)',
        'LightGBM (safe_features)',
        'Isolation Forest (safe_features)',
    ]
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for ax, (model_name, values) in zip(axes, confusion_items.items()):
    cm = confusion_matrix(values['y_true_any'], values['y_pred_any'], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=['Binh thuong', 'Bat thuong'], yticklabels=['Binh thuong', 'Bat thuong'], cbar=False, ax=ax)
    ax.set_title(f'{model_name}\n{values["split"]} - any-anomaly so voi nhan gia')
    ax.set_xlabel('Du doan (co bat thuong nao khong)')
    ax.set_ylabel('Nhan gia phien')

for ax in axes[len(confusion_items):]:
    ax.axis('off')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'output_group_confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()


## 13. Session anomaly report va export

`session_anomaly_tags.csv`: tag rule-level cho moi session. `session_anomaly_report.csv`: tong hop session bi rule hoac model danh dau, kem cot `*_pred_types` liet ke cac anomaly type tung mo hinh du doan. `group_per_label_f1.csv`: F1 theo tung anomaly type.

In [ ]:
model_pred_cols = ['xgboost_pred', 'decision_tree_pred', 'random_forest_pred', 'lightgbm_pred', 'isolation_forest_pred']
model_pred_type_cols = ['xgboost_pred_types', 'decision_tree_pred_types', 'random_forest_pred_types', 'lightgbm_pred_types']
sessions['model_vote_count'] = sessions[model_pred_cols].sum(axis=1)
sessions['is_rule_anomaly'] = sessions['is_anomaly_rule']
sessions['is_model_anomaly'] = (sessions['model_vote_count'] > 0).astype(int)
sessions['is_anomaly_final'] = ((sessions['is_rule_anomaly'] == 1) | (sessions['is_model_anomaly'] == 1)).astype(int)
sessions['rule_anomaly_types'] = sessions['anomaly_types']
sessions['final_anomaly_reason'] = np.select(
    [
        (sessions['is_rule_anomaly'] == 1) & (sessions['is_model_anomaly'] == 1),
        sessions['is_rule_anomaly'] == 1,
        sessions['is_model_anomaly'] == 1,
    ],
    ['Business rules + ML models', 'Business rules', 'ML model-only suspicious'],
    default='Normal',
)

isolation_forest_score_p99 = sessions['isolation_forest_score'].quantile(0.99)

def classify_severity(row):
    if row['total_flags'] >= 3 or row['model_vote_count'] >= 4 or row['isolation_forest_score'] >= isolation_forest_score_p99:
        return 'CRITICAL'
    if row['total_flags'] >= 2 or row['model_vote_count'] >= 2:
        return 'HIGH'
    if row['total_flags'] == 1 or row['model_vote_count'] == 1:
        return 'MEDIUM'
    return 'LOW'

sessions['severity'] = sessions.apply(classify_severity, axis=1)
severity_order = {'CRITICAL': 3, 'HIGH': 2, 'MEDIUM': 1, 'LOW': 0}
sessions['severity_rank'] = sessions['severity'].map(severity_order)

if not (sessions['anomaly_type_count'] == sessions['total_flags']).all():
    raise ValueError('anomaly_type_count phai bang total_flags truoc khi export')
if not (sessions.loc[sessions['is_rule_anomaly'] == 0, 'rule_anomaly_types'] == 'Normal').all():
    raise ValueError('Session binh thuong phai co rule_anomaly_types = Normal truoc khi export')

session_anomaly_tags = sessions[
    ['visitorid', 'session_number', 'is_anomaly', 'anomaly_types', 'anomaly_type_count', 'total_flags'] + flag_cols
].reset_index()

report_cols = [
    'visitorid', 'session_number', 'session_start', 'session_end',
    'is_anomaly_final', 'final_anomaly_reason', 'is_rule_anomaly', 'rule_anomaly_types',
    'is_model_anomaly', 'model_vote_count', 'severity', 'anomaly_type_count', 'total_flags',
    'total_events', 'unique_items', 'session_duration_sec', 'events_per_minute',
    'event_type_entropy', 'hour_entropy', 'xgboost_score', 'decision_tree_score',
    'random_forest_score', 'lightgbm_score', 'isolation_forest_score',
] + model_pred_cols + model_pred_type_cols + flag_cols

session_anomaly_report = (
    sessions[sessions['is_anomaly_final'] == 1]
    .sort_values(['severity_rank', 'model_vote_count', 'total_flags', 'isolation_forest_score'], ascending=[False, False, False, False])
    [report_cols]
    .reset_index()
)

# Bang F1 per-label de nop kem
per_label_export = pd.DataFrame(per_label_records)
per_label_export = per_label_export[
    (per_label_export['split'] == 'test')
]

session_anomaly_tags.to_csv(OUTPUT_DIR / 'session_anomaly_tags.csv', index=False)
session_anomaly_report.to_csv(OUTPUT_DIR / 'session_anomaly_report.csv', index=False)
model_metrics.to_csv(OUTPUT_DIR / 'group_model_metrics.csv', index=False)
per_label_export.to_csv(OUTPUT_DIR / 'group_per_label_f1.csv', index=False)
leakage_audit_df.to_csv(OUTPUT_DIR / 'group_leakage_audit.csv', index=False)
anomaly_type_breakdown.to_csv(OUTPUT_DIR / 'group_anomaly_type_breakdown.csv', index=False)

print(f'So session trong anomaly report: {len(session_anomaly_report):,}')
print('Da export: session_anomaly_tags.csv')
print('Da export: session_anomaly_report.csv')
print('Da export: group_model_metrics.csv')
print('Da export: group_per_label_f1.csv (F1 tung anomaly type)')
print('Da export: group_leakage_audit.csv')
print('Da export: group_anomaly_type_breakdown.csv')
print('\nVi du session anomaly report:')
display(session_anomaly_report.head(15))


## 14. Ket luan bao cao

- Don vi phan tich la `session_id` (gap 30 phut).
- **Cac mo hinh giam sat gio du doan dong thoi 12 anomaly type (multi-label)** thay vi nhi phan, dung dung voi thiet ke gan nhan da nhan o muc 4.
- Decision Tree & Random Forest multi-output truc tiep; XGBoost & LightGBM qua `MultiOutputClassifier`.
- Metric chinh la macro/micro-F1 va Hamming loss tren safe features; bang full features chi minh hoa rule-mimic.
- F1 phan hoa giua cac type: type con proxy trong safe set thi khoi phuc tot, type bi cat dac trung (vd Night crawler) thi rat thap - day la ket qua dang phan tich.
- Isolation Forest van khong giam sat, chi cho overlap nhi phan voi pseudo-label.
- RetailRocket khong co nhan that, moi danh gia la tren pseudo-label do nhom thiet ke.

## 15. Tài liệu tham khảo (cơ sở cho ngưỡng và đặc trưng)

1. Catledge, L. D., & Pitkow, J. E. (1995). *Characterizing browsing strategies in the World-Wide Web.* Computer Networks and ISDN Systems, 27(6), 1065–1073. — nguồn gốc ngưỡng session ~25.5→30 phút.
2. Cooley, R., Mobasher, B., & Srivastava, J. (1999). *Data preparation for mining World Wide Web browsing patterns.* Knowledge and Information Systems, 1(1), 5–32. — heuristic sessionization 30 phút.
3. Tan, P.-N., & Kumar, V. (2002). *Discovery of Web Robot Sessions Based on their Navigational Patterns.* Data Mining and Knowledge Discovery, 6(1), 9–35. — phát hiện bot bằng đặc trưng cấp phiên (số request, inter-arrival time, độ phủ item), cây C4.5 ~90% accuracy.
4. Doran, D., & Gokhale, S. S. (2011). *Web robot detection techniques: overview and limitations.* Data Mining and Knowledge Discovery, 22(1), 183–210. — tương phản thời gian/độ dài phiên giữa robot và người.
5. Google Analytics — *About Analytics sessions* (mặc định 30 phút không hoạt động kết thúc phiên).
6. Baymard Institute (2025). *Cart Abandonment Rate Statistics* — tỉ lệ bỏ giỏ trung bình ~70.22% (meta-analysis 50 nghiên cứu).

*Lưu ý diễn giải:* bảng chính đo khả năng mô hình **tái tạo lại bộ business-rule** từ rule-aligned features (một mục tiêu hợp lệ, cho chỉ số cao thật), KHÔNG phải năng lực phát hiện gian lận thật — vì RetailRocket không có nhãn thật. Bảng đối chứng safe-features được giữ để minh bạch về label leakage.

In [ ]:
print('=' * 70)
print('TONG KET DO AN NHOM - CAP SESSION (MULTI-LABEL)')
print('=' * 70)
print('Dataset: RetailRocket E-commerce events')
print(f'So su kien: {len(events):,}')
print(f'So visitor: {events["visitorid"].nunique():,}')
print(f'So session profile: {len(sessions):,}')
print(f'So anomaly type duoc mo hinh hoa: {len(model_label_cols)}/{len(flag_cols)}')
print(f'Bat thuong theo business rules: {sessions["is_rule_anomaly"].sum():,} ({sessions["is_rule_anomaly"].mean() * 100:.2f}%)')
print(f'Bat thuong theo ket qua tong hop: {sessions["is_anomaly_final"].sum():,} ({sessions["is_anomaly_final"].mean() * 100:.2f}%)')
print('\nBang metric chinh (multi-label, macro/micro F1):')
display(main_metrics)
print('\nBang phu full features hoc lai luat:')
display(leakage_control_metrics)
print('\nF1 theo tung anomaly type (test, safe features):')
if not per_label_test.empty:
    display(f1_pivot)
print('\nBreakdown anomaly type:')
display(anomaly_type_breakdown)
print('\nPhan bo muc do nghiem trong trong report:')
display(session_anomaly_report['severity'].value_counts().rename_axis('severity').reset_index(name='count'))
